# Check the GPU

In [1]:
import torch
import platform

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())

    print("\nGPU properties:")
    print(torch.cuda.get_device_properties(0))
else:
    print("No GPU detected. Please check runtime settings.")

Python version: 3.13.16
PyTorch version: 2.14.1+cu130

CUDA available: True
GPU name: Tesla T4
CUDA version: 13.0
GPU count: 1

GPU properties:
_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=14912MB, multi_processor_count=40, uuid=7fc7f93f-b76a-86ce-7f26-dc29f0e8ee57, pci_bus_id=0, pci_device_id=4, pci_domain_id=0, L2_cache_size=4MB)


## Transfer the dataset and model
We need these two files from your Windows project:
- data/processed/har_preprocessed.pt
- models/har_cnn_baseline.pth
We'll upload them to the Colab session.

In [10]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded:
    print(filename)

Saving har_preprocessed.pt to har_preprocessed.pt

Uploaded files:
har_preprocessed.pt


## Verify the uploaded files

In [11]:
import os

for filename in [
    "har_preprocessed.pt",
    "har_cnn_baseline.pth"
]:
    print(filename, ":", os.path.exists(filename))

har_preprocessed.pt : True
har_cnn_baseline.pth : True


## Load the dataset and checkpoint

In [12]:
# Load the preprocessed dataset
data = torch.load(
    "har_preprocessed.pt",
    map_location="cpu",
    weights_only=False
)

# Load the trained model checkpoint
checkpoint = torch.load(
    "har_cnn_baseline.pth",
    map_location="cpu",
    weights_only=True
)

print("Dataset keys:", data.keys())
print("\nCheckpoint keys:", checkpoint.keys())

print("\nModel configuration:")
print("Architecture:", checkpoint["architecture"])
print("Number of classes:", checkpoint["num_classes"])
print("Input channels:", checkpoint["input_channels"])
print("Sequence length:", checkpoint["sequence_length"])
print("Training epochs:", checkpoint["epochs"])
print("Original test accuracy:", checkpoint["test_accuracy"])

Dataset keys: dict_keys(['X_train', 'X_test', 'y_train', 'y_test', 'train_indices', 'val_indices'])

Checkpoint keys: dict_keys(['model_state_dict', 'architecture', 'num_classes', 'input_channels', 'sequence_length', 'batch_size', 'epochs', 'test_loss', 'test_accuracy'])

Model configuration:
Architecture: HAR_CNN
Number of classes: 6
Input channels: 6
Sequence length: 128
Training epochs: 21
Original test accuracy: 0.8920936545639634


## Reconstruct the trained model

In [13]:
import torch.nn as nn

class HAR_CNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()

        self.conv1 = nn.Conv1d(6, 32, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 32, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)

        return x


# Reconstruct model and load trained weights
model = HAR_CNN(num_classes=checkpoint["num_classes"])

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

total_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Model loaded successfully.")
print("Architecture:", checkpoint["architecture"])
print("Trainable parameters:", f"{total_parameters:,}")
print("Model mode:", "Evaluation" if not model.training else "Training")

Model loaded successfully.
Architecture: HAR_CNN
Trainable parameters: 274,342
Model mode: Evaluation


## Prepare the GPU test DataLoader

In [14]:
from torch.utils.data import DataLoader, TensorDataset

# Prepare test tensors on CPU
X_test_tensor = data["X_test"].detach().clone().to(torch.float32)
y_test_tensor = data["y_test"].detach().clone().to(torch.long)

# Create test dataset
BATCH_SIZE = 64

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Inspect the dataset
signals, labels = next(iter(test_loader))

print("Test samples:", len(test_dataset))
print("Number of batches:", len(test_loader))
print("Batch size:", BATCH_SIZE)
print("Signal batch shape:", signals.shape)
print("Label batch shape:", labels.shape)
print("Signal device:", signals.device)
print("Label device:", labels.device)

Test samples: 2947
Number of batches: 47
Batch size: 64
Signal batch shape: torch.Size([64, 6, 128])
Label batch shape: torch.Size([64])
Signal device: cpu
Label device: cpu


## GPU inference benchmark

In [17]:
import time
import numpy as np

# Select GPU
device = torch.device("cuda")

model = model.to(device)
model.eval()

WARMUP_BATCHES = 5
NUM_RUNS = 3


def benchmark_gpu_inference(model, data_loader, device, warmup_batches=5):

    model.eval()

    # Warm-up GPU
    with torch.inference_mode():
        for i, (signals, _) in enumerate(data_loader):
            if i >= warmup_batches:
                break

            signals = signals.to(device)
            _ = model(signals)

    # Ensure warm-up operations are finished
    torch.cuda.synchronize()

    run_times = []

    # Actual benchmark
    for run in range(NUM_RUNS):

        torch.cuda.synchronize()
        start_time = time.perf_counter()

        with torch.inference_mode():
            for signals, _ in data_loader:
                signals = signals.to(device)
                _ = model(signals)

        # Wait for all GPU operations to finish
        torch.cuda.synchronize()

        end_time = time.perf_counter()

        elapsed_time = end_time - start_time
        run_times.append(elapsed_time)

        print(f"Run {run + 1}: {elapsed_time:.4f} seconds")

    # Calculate performance
    average_time = np.mean(run_times)

    return {
        "device": "cuda",
        "gpu_name": torch.cuda.get_device_name(0),
        "average_time_seconds": average_time,
        "average_batch_time_ms": (
            average_time / len(data_loader) * 1000
        ),
        "throughput_samples_per_second": (
            len(data_loader.dataset) / average_time
        )
    }


gpu_results = benchmark_gpu_inference(
    model,
    test_loader,
    device
)

print("\nGPU Inference Benchmark Results")
print("--------------------------------")
for key, value in gpu_results.items():
    print(f"{key}: {value}")

Run 1: 0.0811 seconds
Run 2: 0.0532 seconds
Run 3: 0.0450 seconds

GPU Inference Benchmark Results
--------------------------------
device: cuda
gpu_name: Tesla T4
average_time_seconds: 0.05974692600011622
average_batch_time_ms: 1.2712111914918345
throughput_samples_per_second: 49324.71337511603


## Save the GPU benchmark results

In [18]:
import pandas as pd

gpu_results_df = pd.DataFrame([{
    **gpu_results,
    "gpu_memory_mb": (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
    ),
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "batch_size": BATCH_SIZE,
    "test_samples": len(test_dataset),
    "warmup_batches": WARMUP_BATCHES,
    "benchmark_runs": NUM_RUNS
}])

gpu_results_df.to_csv(
    "inference_benchmark_gpu.csv",
    index=False
)

display(gpu_results_df)

,device,gpu_name,average_time_seconds,average_batch_time_ms,throughput_samples_per_second,gpu_memory_mb,pytorch_version,cuda_version,batch_size,test_samples,warmup_batches,benchmark_runs
0,cuda,Tesla T4,0.059747,1.271211,49324.713375,14912.6875,2.14.1+cu130,13.0,64,2947,5,3


In [19]:
from google.colab import files

files.download("inference_benchmark_gpu.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>